Voice measures in early Parkinson’s disease” (Little et al., 2007)

In [22]:
!pip install praat-parselmouth librosa numpy pandas soundfile
import parselmouth
from parselmouth.praat import call
import numpy as np
import pandas as pd
from google.colab import files

In [23]:
uploaded = files.upload()


Saving harvard.wav to harvard (1).wav


In [24]:
file_path = list(uploaded.keys())[0]


In [26]:
def extract_voice_features(filepath):
    """
    Extracts 22 Parkinson's acoustic voice features from a .wav file.
    Returns a one-row pandas DataFrame.
    """
    snd = parselmouth.Sound(filepath)

    # --- Pitch statistics ---
    pitch = call(snd, "To Pitch", 0.0, 75, 500)
    mean_F0 = call(pitch, "Get mean", 0, 0, "Hertz")
    min_F0  = call(pitch, "Get minimum", 0, 0, "Hertz", "Parabolic")
    max_F0  = call(pitch, "Get maximum", 0, 0, "Hertz", "Parabolic")

    # --- Point process for jitter/shimmer ---
    pointProcess = call(snd, "To PointProcess (periodic, cc)", 75, 500)

    # Jitter metrics
    jitter_local = call(pointProcess, "Get jitter (local)", 0, 0, 75, 500, 1.3)
    jitter_abs   = call(pointProcess, "Get jitter (local, absolute)", 0, 0, 75, 500, 1.3)
    jitter_rap   = call(pointProcess, "Get jitter (rap)", 0, 0, 75, 500, 1.3)
    jitter_ppq5  = call(pointProcess, "Get jitter (ppq5)", 0, 0, 75, 500, 1.3)
    jitter_ddp   = call(pointProcess, "Get jitter (ddp)", 0, 0, 75, 500, 1.3)

    # Shimmer metrics
    shimmer_local  = call([snd, pointProcess], "Get shimmer (local)", 0, 0, 75, 500, 1.3, 1.6)
    shimmer_db     = call([snd, pointProcess], "Get shimmer (local_dB)", 0, 0, 75, 500, 1.3, 1.6)
    shimmer_apq3   = call([snd, pointProcess], "Get shimmer (apq3)", 0, 0, 75, 500, 1.3, 1.6)
    shimmer_apq5   = call([snd, pointProcess], "Get shimmer (apq5)", 0, 0, 75, 500, 1.3, 1.6)
    shimmer_apq11  = call([snd, pointProcess], "Get shimmer (apq11)", 0, 0, 75, 500, 1.3, 1.6)
    shimmer_dda    = call([snd, pointProcess], "Get shimmer (dda)", 0, 0, 75, 500, 1.3, 1.6)

    # --- Harmonicity / Noise ---
    harmonicity = call(snd, "To Harmonicity (cc)", 0.01, 75, 0.1, 1.0)
    hnr = call(harmonicity, "Get mean", 0, 0)
    nhr = 1 / (hnr + 1e-6)  # approximate inverse ratio

    # --- Nonlinear dynamics & entropy (placeholders) ---
    rpde = np.nan
    dfa = np.nan
    spread1 = np.nan
    spread2 = np.nan
    d2 = np.nan
    ppe = np.nan

    # --- Combine all features into a dictionary ---
    data = {
        "MDVP:Fo(Hz)": mean_F0,
        "MDVP:Fhi(Hz)": max_F0,
        "MDVP:Flo(Hz)": min_F0,
        "MDVP:Jitter(%)": jitter_local * 100,
        "MDVP:Jitter(Abs)": jitter_abs,
        "MDVP:RAP": jitter_rap,
        "MDVP:PPQ": jitter_ppq5,
        "Jitter:DDP": jitter_ddp,
        "MDVP:Shimmer": shimmer_local,
        "MDVP:Shimmer(dB)": shimmer_db,
        "Shimmer:APQ3": shimmer_apq3,
        "Shimmer:APQ5": shimmer_apq5,
        "MDVP:APQ": shimmer_apq11,
        "Shimmer:DDA": shimmer_dda,
        "NHR": nhr,
        "HNR": hnr,
        "RPDE": rpde,
        "DFA": dfa,
        "spread1": spread1,
        "spread2": spread2,
        "D2": d2,
        "PPE": ppe,
    }

    # Return as DataFrame
    df = pd.DataFrame([data])
    return df

In [27]:
voice_df = extract_voice_features(file_path)

In [28]:
print(voice_df.head())

   MDVP:Fo(Hz)  MDVP:Fhi(Hz)  MDVP:Flo(Hz)  MDVP:Jitter(%)  MDVP:Jitter(Abs)  \
0   133.832336    489.708489     74.847612             NaN               NaN   

   MDVP:RAP  MDVP:PPQ  Jitter:DDP  MDVP:Shimmer  MDVP:Shimmer(dB)  ...  \
0       NaN       NaN         NaN           NaN               NaN  ...   

   MDVP:APQ  Shimmer:DDA       NHR       HNR  RPDE  DFA  spread1  spread2  D2  \
0       NaN          NaN  0.103089  9.700365   NaN  NaN      NaN      NaN NaN   

   PPE  
0  NaN  

[1 rows x 22 columns]
